## 📝 Instrucciones

Sistema de detección de enlaces spam
Queremos implementar un sistema que sea capaz de detectar automáticamente si una página web contiene spam o no basándonos en su URL.

Paso 1: Carga del conjunto de datos
El conjunto de datos se puede encontrar en esta carpeta de proyecto bajo el nombre url_spam.csv. Puedes cargarlo en el código directamente desde el siguiente enlace:

https://breathecode.herokuapp.com/asset/internal-link?id=932&path=url_spam.csv


O descargarlo y añadirlo a mano en tu repositorio.

In [ ]:
import pandas as pd
import re
from nltk import download
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report
from joblib import dump

download("wordnet")
download("stopwords")

lemmatizer = WordNetLemmatizer()
stop_words = stopwords.words("english")

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\ekbal\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ekbal\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [4]:
df = pd.read_csv("../data/raw/url_spam.csv", sep=",")
df.head()

,url,is_spam
0,https://briefingday.us8.list-manage.com/unsubs...,True
1,https://www.hvper.com/,True
2,https://briefingday.com/m/v4n3i4f3,True
3,https://briefingday.com/n/20200618/m#commentform,False
4,https://briefingday.com/fan,True


## Paso 2: Preprocesa los enlaces


Utiliza lo visto en este módulo para transformar los datos para compatibilizarlos con el modelo que queremos entrenar. Segmenta las URLs en partes según sus signos de puntuación, elimina las stopwords, lematiza, etcétera.

Asegúrate de dividir convenientemente el conjunto de datos en train y test como hemos visto en lecciones anteriores.

In [8]:
def preprocess_url(url):
    url = url.lower()
    url = re.sub(r'https?://', '', url)
    
    tokens = re.split(r'[\/\.\-\_\=\?\&\#\:\@\%\+\~]', url)
    
    tokens = [t for t in tokens if len(t) > 2]
    
    tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in stop_words]
    
    return tokens

df["tokens"] = df["url"].apply(preprocess_url)

tokens_list = [" ".join(tokens) for tokens in df["tokens"]]

vectorizer = TfidfVectorizer(max_features=5000, max_df=0.8, min_df=5)
X = vectorizer.fit_transform(tokens_list).toarray()
y = df["is_spam"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Train size: {X_train.shape}")
print(f"Test size:  {X_test.shape}")

Train size: (2399, 746)
Test size:  (600, 746)


## Paso 3: Construye un SVM

Comienza a resolver el problema implementando un SVM con los parámetros por defecto. Entrénalo y analiza sus resultados.

In [10]:
model = SVC(random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["No spam", "Spam"]))

Accuracy: 0.9483

Classification Report:
              precision    recall  f1-score   support

     No spam       0.95      0.99      0.97       455
        Spam       0.95      0.83      0.89       145

    accuracy                           0.95       600
   macro avg       0.95      0.91      0.93       600
weighted avg       0.95      0.95      0.95       600



## Paso 4: Optimiza el modelo anterior


Después de entrenar el SVM, optimiza sus hiperparámetros utilizando un grid search o un random search.

In [12]:
param_grid = {
    "kernel": ["linear", "rbf"],
    "C": [0.1, 1, 10],
    "gamma": ["scale", "auto"]
}

grid_search = GridSearchCV(
    estimator=SVC(random_state=42),
    param_grid=param_grid,
    scoring="f1",
    cv=5,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print(f"Mejores parámetros: {grid_search.best_params_}")
print(f"Mejor puntuación F1: {grid_search.best_score_:.4f}")

best_model = grid_search.best_estimator_
y_pred_best = best_model.predict(X_test)

print(f"\nAccuracy: {accuracy_score(y_test, y_pred_best):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_best, target_names=["No spam", "Spam"]))

Fitting 5 folds for each of 12 candidates, totalling 60 fits
Mejores parámetros: {'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}
Mejor puntuación F1: 0.8951

Accuracy: 0.9550

Classification Report:
              precision    recall  f1-score   support

     No spam       0.96      0.98      0.97       455
        Spam       0.95      0.86      0.90       145

    accuracy                           0.95       600
   macro avg       0.95      0.92      0.94       600
weighted avg       0.95      0.95      0.95       600



### Comparación entre modelos:

| Métrica | SVM base | SVM optimizado |
|---|---|---|
| Accuracy | 94.83% | 95.50% |
| Spam Recall | 0.83 | 0.86 |
| Spam F1 | 0.89 | 0.90 |
| Macro F1 | 0.93 | 0.94 |

## Paso 5: Guarda el modelo


Almacena el modelo en la carpeta correspondiente

In [14]:
dump(best_model, "../models/url_spam/svm_url_spam_model.joblib")
dump(vectorizer, "../models/url_spam/tfidf_vectorizer.joblib")

['../models/url_spam/tfidf_vectorizer.joblib']